# End-to-end RMY generation from a NOAA station number

Enter one NOAA station number and this notebook builds, for that station:

1. **AMYs**: one actual meteorological year EPW per year for the **15 most recent complete years**, from NOAA station observations and NSRDB satellite solar radiation (PVGIS outside the Americas). Gaps are filled with documented, flagged methods.
2. **TMY**: a typical meteorological year from those AMYs, following ISO 15927-4.
3. **RMY**: the representative meteorological year, with the peak year's heat waves and cold spells inserted into the TMY (Tarkhan et al. 2025).

and plots the result. The 15-year period moves forward automatically as NOAA publishes new complete years.

**You need** a free NREL API key and the email it is registered to, for stations in the Americas (<a href="https://developer.nlr.gov/docs/climate/guide/" target="_blank">get a key here</a>). Add them in Colab's **Secrets** panel (🔑 on the left) as `NSRDB_API_KEY` and `NSRDB_EMAIL`, with notebook access on; otherwise section 1.0 asks for them. Keys are never printed or saved in the outputs.

**Run time**: roughly 1-3 minutes per year (downloads), so about 20-40 minutes for a station.

Outputs, in `/content/output/<City>/` and as a zip at the end:

```
AMY/           one AMY EPW per year
AMY_reports/   per-year reports, hourly gap-filling provenance, years set aside as flagged
TMY/           the ISO 15927-4 TMY and the source year of each month
RMY/           the RMY EPW, heat-wave and cold-spell CSVs, event timeline, log
```

The notebook for the 8 cities already in the repository is `RMY_Generation_Colab.ipynb`.

In [ ]:
# @title Settings { display-mode: "form" }
# @markdown **NOAA station number**: 6-character USAF/WMO or 11-character USAF+WBAN, for example `72509014739` (Boston Logan). Find it at [climate.onebuilding.org](https://climate.onebuilding.org/) or in NOAA's [station list](https://www.ncei.noaa.gov/pub/data/noaa/isd-history.csv).
STATION_NUMBER = "72509014739"  # @param {type:"string"}

STATION_NUMBER = STATION_NUMBER.strip().replace("-", "")
SETTINGS_APPLIED = True
print(f"✅ Station {STATION_NUMBER} (the city name is found automatically)")

# **0.0 Setup**

Installs missing packages and makes a fast copy of the code (latest version only, code folders only).

In [ ]:
import importlib.util, subprocess, sys

needed = {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
          "requests": "requests", "timezonefinder": "timezonefinder",
          "geonamescache": "geonamescache",
          "h5py": "h5py", "fsspec": "fsspec", "s3fs": "s3fs"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("Installed:", ", ".join(missing) if missing else "nothing (all packages already available)")

%cd /content
!rm -rf RMY
!git clone -q --depth 1 --single-branch --branch main --filter=blob:none --sparse https://github.com/Nadatarkhan/RMY.git
%cd /content/RMY
!git sparse-checkout set rmy tmy amy_sourcing
!git log -1 --format='Running commit %h: %s'

In [ ]:
# Load the three steps (forget any copy loaded earlier in this session).
import importlib.util, sys
from pathlib import Path

REPO = Path("/content/RMY")
for name in [m for m in sys.modules if m == "rmy" or m.startswith("rmy.")]:
    del sys.modules[name]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

def load(name, path):
    spec = importlib.util.spec_from_file_location(name, REPO / path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

amy_sourcing = load("generate_amys", "amy_sourcing/generate_amys.py")
tmy_code = load("generate_tmy_from_amys", "tmy/generate_tmy_from_amys.py")
from rmy import run_full_rmy_pipeline
from rmy.utils import read_epw_file
from rmy.visualization import visualize_extreme_events
print("✅ Code loaded")

# **1.0 NREL API key**

Read from Colab Secrets (`NSRDB_API_KEY`, `NSRDB_EMAIL`) or, if absent, asked for here without being shown. Only stations in the Americas need it.

In [ ]:
import os
from getpass import getpass

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

NSRDB_API_KEY = secret("NSRDB_API_KEY")
NSRDB_EMAIL = secret("NSRDB_EMAIL")
if not (NSRDB_API_KEY and NSRDB_EMAIL):
    print("No NSRDB_API_KEY / NSRDB_EMAIL secrets found. Enter them below (press Enter to skip for stations outside the Americas).")
    NSRDB_API_KEY = NSRDB_API_KEY or getpass("NREL API key: ").strip() or None
    NSRDB_EMAIL = NSRDB_EMAIL or getpass("Email registered with that key: ").strip() or None
print("✅ NREL key available" if NSRDB_API_KEY and NSRDB_EMAIL else "No NREL key: only stations outside the Americas will work.")

# **2.0 AMYs: the 15 most recent complete years**

The last complete year is read from the NOAA station record. A year that fails, or has a gap in temperature, dew point or pressure too long to fill (flagged and kept in `AMY_reports/flagged`), is replaced by the year before.

In [ ]:
import pandas as pd
from IPython.display import display

if not globals().get("SETTINGS_APPLIED"):
    raise RuntimeError("Run the Settings cell at the top first.")
OUTPUT_DIR = Path("/content/output")
AMY_RUN = amy_sourcing.generate_amys(STATION_NUMBER, OUTPUT_DIR, NSRDB_API_KEY, NSRDB_EMAIL)
CITY = AMY_RUN["city"]
CITY_DIR = OUTPUT_DIR / CITY
AMY_DIR = AMY_RUN["amy_dir"]
YEARS = AMY_RUN["years"]

cols = ["year", "status", "qc_grade", "radiation_provider_selected", "DB_missing_hours", "DB_max_gap_hours",
        "wind_direction_remaining_missing_hours", "total_sky_still_missing_hours", "error"]
manifest = AMY_RUN["manifest"]
display(manifest[[c for c in cols if c in manifest.columns]])

# **3.0 TMY (ISO 15927-4)**

For each calendar month, the year whose daily temperature, humidity and solar radiation are most typical of all the AMY years (ISO 15927-4). The header comes from the most recent non-leap AMY.

In [ ]:
import calendar, contextlib, io

TMY_DIR = CITY_DIR / "TMY"
header_epw = max((p for p in AMY_DIR.glob("*.epw") if not calendar.isleap(tmy_code.year_from_filename(p))),
                 key=tmy_code.year_from_filename)
station_id = AMY_RUN["station"]["station_id"]
TMY_PATH = TMY_DIR / f"{CITY}_{station_id}_TMY_{YEARS[0]}-{YEARS[-1]}.epw"
with contextlib.redirect_stdout(io.StringIO()):
    tmy_code.generate_tmy_from_amys(AMY_DIR, TMY_PATH, header_epw, years=(YEARS[0], YEARS[-1]))
sources = pd.read_csv(TMY_PATH.with_suffix(".month_sources.csv"))
sources["month"] = sources["month"].map(lambda m: calendar.month_abbr[m])
print("✅ TMY:", TMY_PATH.name)
display(sources[["month", "year", "top3_years"]].rename(
    columns={"year": "selected year", "top3_years": "three most typical years"}).set_index("month").T)

# **4.0 RMY**

Detects heat waves and cold spells in all AMY years (thresholds, GNN and EVT), picks the peak year of each, inserts its events into the TMY and rebalances the monthly means. The full log is saved to `RMY/log.txt`.

In [ ]:
RMY_DIR = CITY_DIR / "RMY"
log = io.StringIO()
with contextlib.redirect_stdout(log):
    RMY_PATH = run_full_rmy_pipeline(AMY_DIR, TMY_DIR, RMY_DIR)
RMY_DIR.mkdir(parents=True, exist_ok=True)
(RMY_DIR / "log.txt").write_text(log.getvalue())
for line in log.getvalue().splitlines():
    if any(k in line for k in ("thresholds (all", "GNN:", "EVT ", "runner-up", "inserted as its most",
                               "events integrated", "smoothing:", "WARNING", "Final RMY")):
        print(line.strip())

TMY = read_epw_file(TMY_PATH)
RMY = read_epw_file(RMY_PATH)
heat = pd.read_csv(RMY_DIR / "hotspells/heatwave_events_peak.csv")
cold = pd.read_csv(RMY_DIR / "coldspells/coldspells_events_peak.csv")
def stats(df):
    t = df.temp_air
    return [f"{t.max():.1f}", f"{t.min():.1f}", f"{t.mean():.1f}", int((t >= 30).sum()), int((t <= -10).sum())]
summary = pd.DataFrame({"TMY": stats(TMY), "RMY": stats(RMY)},
                       index=["Hottest hour (°C)", "Coldest hour (°C)", "Annual mean (°C)", "Hours ≥ 30 °C", "Hours ≤ -10 °C"])
print(f"\nPeak heat-wave year {int(heat.year.iloc[0]) if len(heat) else '-'}: {len(heat)} events | "
      f"peak cold-spell year {int(cold.year.iloc[0]) if len(cold) else '-'}: {len(cold)} events")
display(summary)

# **5.0 Visualizations**

### 5.1 Extreme Event Explorer
Every heat wave and cold spell detected in the AMY years. Circle size is duration; heat waves are colored by heat index, cold spells by wind speed.

In [ ]:
visualize_extreme_events(RMY_DIR / "hotspells/heatwave_events.csv", RMY_DIR / "coldspells/coldspells_events.csv",
                         output_path=RMY_DIR / "event_timeline.png",
                         title=f"{CITY}: heat waves and cold spells {YEARS[0]}-{YEARS[-1]}")

### 5.2 Daily temperature range: RMY vs TMY
Blue band: the TMY's daily minimum to maximum. Orange lines: the RMY's daily maximum and minimum. The RMY departs from the TMY where the peak years' events were inserted.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

TMY_COLOR, RMY_COLOR, GRID = "#2a78d6", "#eb6834", "#e4e3df"

def daily(df):
    g = df.groupby([df.month, df.day]).temp_air
    return g.min().to_numpy(), g.max().to_numpy()

fig, ax = plt.subplots(figsize=(14, 4.5))
x = np.arange(365)
lo, hi = daily(TMY[~((TMY.month == 2) & (TMY.day == 29))])
ax.fill_between(x, lo, hi, color=TMY_COLOR, alpha=0.25, linewidth=0, label="TMY daily range")
lo, hi = daily(RMY[~((RMY.month == 2) & (RMY.day == 29))])
ax.plot(x, hi, color=RMY_COLOR, lw=1.2, label="RMY daily max / min")
ax.plot(x, lo, color=RMY_COLOR, lw=1.2)
starts = np.cumsum([0] + [calendar.monthrange(2001, m)[1] for m in range(1, 12)])
ax.set_xticks(starts + 14, [calendar.month_abbr[m] for m in range(1, 13)])
ax.set_ylabel("Dry-bulb temperature (°C)")
ax.set_title(f"{CITY}: daily temperature range, RMY vs TMY", loc="left")
ax.grid(axis="y", color=GRID); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout(); fig.savefig(RMY_DIR / "daily_range_rmy_vs_tmy.png", dpi=150); plt.show()

### 5.3 The most extreme inserted events, hour by hour
The hottest heat wave and the coldest cold spell of the peak years, with three days either side, in the RMY and the TMY. Shaded: the days taken from the peak year (events longer than 21 days are inserted as their most severe 21 days).

In [ ]:
def event_window(events, key, hottest):
    if events.empty:
        return None
    row = events.loc[events[key].idxmax() if hottest else events[key].idxmin()]
    start = pd.to_datetime(row.begin_date, dayfirst=True) - pd.Timedelta(days=3)
    end = pd.to_datetime(row.end_date, dayfirst=True) + pd.Timedelta(days=3)
    return row, start, end

panels = [w for w in (event_window(heat, "max_tmax", True), event_window(cold, "min_tmin", False)) if w]
fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 4), squeeze=False)
for ax, (row, start, end), kind in zip(axes[0], panels, ("Heat wave", "Cold spell")):
    days = pd.date_range(start, end, freq="D")
    keys = set(zip(days.month, days.day))
    for df, color, label in ((TMY, TMY_COLOR, "TMY"), (RMY, RMY_COLOR, "RMY")):
        sel = df[[k in keys for k in zip(df.month, df.day)]]
        ax.plot(np.arange(len(sel)) / 24, sel.temp_air.to_numpy(), color=color, lw=2, label=label)
    # Shade the days whose hours are the peak year's (inserted days).
    src = read_epw_file(next(p for p in AMY_DIR.glob("*.epw") if tmy_code.year_from_filename(p) == int(row.year)))
    labelled = False
    for k, d in enumerate(days):
        r = RMY[(RMY.month == d.month) & (RMY.day == d.day)].temp_air.to_numpy()
        a = src[(src.month == d.month) & (src.day == d.day)].temp_air.to_numpy()
        if len(r) == len(a) and np.allclose(r, a):
            ax.axvspan(k, k + 1, color=RMY_COLOR, alpha=0.15, lw=0,
                       label=None if labelled else f"days from {int(row.year)}")
            labelled = True
    ax.set_title(f"{kind} {pd.to_datetime(row.begin_date, dayfirst=True):%d %b}-"
                 f"{pd.to_datetime(row.end_date, dayfirst=True):%d %b %Y}, ±3 days", loc="left")
    ax.set_xlabel(f"Days from {start:%d %b}"); ax.set_ylabel("Dry-bulb temperature (°C)")
    ax.grid(axis="y", color=GRID); ax.spines[["top", "right"]].set_visible(False); ax.legend(frameon=False)
plt.tight_layout(); fig.savefig(RMY_DIR / "peak_events_rmy_vs_tmy.png", dpi=150); plt.show()

# **6.0 Download**

All outputs for this station as one zip (the download cache is left out).

In [ ]:
import shutil, tempfile
from google.colab import files

staging = Path(tempfile.mkdtemp()) / CITY
shutil.copytree(CITY_DIR, staging, ignore=shutil.ignore_patterns("_cache"))
archive = shutil.make_archive(f"/content/{CITY}_AMY_TMY_RMY", "zip", staging.parent, CITY)
print("Zip:", archive)
files.download(archive)